In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
file_path = r"Q:\sachuriga\Sachuriga_Python/quattrocolo-nwb4fp/ASSY-236-F.prb"

# Read the file and parse the dictionary
local_vars = {'np': np}
with open(file_path, 'r') as file:
    exec(file.read(), local_vars)  # Execute the file content with NumPy in scope

channel_groups = local_vars.get('channel_groups')
if channel_groups is None:
    raise ValueError(f"'channel_groups' not found in {file_path}")

# Assuming channel_groups is loaded from Step 1
data = []
for group_id, group_data in channel_groups.items():
    channels = group_data['channels']
    geometry = group_data['geometry']
    for channel in channels:
        x, y = geometry[channel]
        data.append({
            'group_id': group_id,
            'channel_id': channel,
            'x': x,
            'y': y
        })
probe_df = pd.DataFrame(data)

import os
def get_pkl_files(folder_path):
    # List all files in the directory
    all_files = os.listdir(folder_path)
    # Filter files that end with "withDLC.pkl"
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

# Example usage
folder_path = r"S:\Sachuriga\file_with_table\ripple_ch"  # Replace with your actual folder path
adjust_path = r"S:\Sachuriga\file_with_table\ripple_ch"
new_files = []
pkl_files = get_pkl_files(folder_path)
for file in pkl_files:

    df_unit_table = pd.read_pickle(fr'{folder_path}/{file}').reset_index(drop=True)
    # df_unit_table=dff[dff['unit_quality']=="good"]
    # #df_unit_table = df['unit_table'][0]
    # df_unit_table=dff.reset_index(drop=True)
    
    new_y = []
    group = []
    for i in range(len(df_unit_table)):
        group.append(probe_df[probe_df['channel_id']==df_unit_table['ch'][i]]['group_id'].values[0])
        
    df_unit_table['group'] = group
    unique_groups = df_unit_table['group'].unique()  # Re
    df_list = [df_unit_table[df_unit_table['group'] == group].reset_index(drop=True) for group in unique_groups]

    df_list_actual=[]
    for i, df in enumerate(df_list):
   
        median_value = np.median(df[df['cell_type']== "pyramidal"]['y'])
        mean_value = np.mean(df[df['cell_type']== "pyramidal"]['y'])
        y_pos=[]
        y_pos_mean=[]
        idx=0
        y_pos_r1 = []
        y_pos_r2 = []
        y_pos_r3 = []
        y_sub=[]
        for i in df['y']:
            if df['ripple_ch'][idx]:
                r1 = df['ripple_ch'][idx]
                r2 = df['ripple_ch_3std'][idx]
                r3 = df['ripple_ch_4std'][idx]
                y1 = probe_df[probe_df['channel_id']==r1]['y'].values
                y2 = probe_df[probe_df['channel_id']==r2]['y'].values
                y3 = probe_df[probe_df['channel_id']==r2]['y'].values
                yr1 = i-y1
                yr2 = i-y2
                yr3 = i-y3
                y_pos_r1.append(yr1[0])
                y_pos_r2.append(yr2[0])
                y_pos_r3.append(yr3[0])
            else:
                y_pos_r1.append(None)
                y_pos_r2.append(None)
                y_pos_r3.append(None)
            
            y_sub.append('deep' if yr2[0]>0 else 'superficial')
            y_pos.append(i-median_value+75)
            y_pos_mean.append(i-mean_value+75)

 
            idx += 0
        df['addjust y'] = y_pos_mean
        df['addjust y median'] = y_pos
        df['addjust y r1'] = y_pos_r1
        df['addjust y r2'] = y_pos_r2
        df['addjust y r3'] = y_pos_r3
        df['sub_population'] = y_sub
        df_list_actual.append(df)
    df_list_actual = pd.concat(df_list_actual, axis=0)
    if os.path.exists(fr"{adjust_path}/{file}"):
        os.remove(fr"{adjust_path}/{file}")
    df_list_actual.to_pickle(fr"{adjust_path}/{file}")

In [ ]:
import os
import pandas as pd
def get_pkl_files(folder_path):
    # List all files in the directory
    all_files = os.listdir(folder_path)
    # Filter files that end with "withDLC.pkl"
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']

folder_path = fr"S:\Sachuriga\file_with_table\ripple_ch\Functional_connections"
df_good = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')

df_good['connectivity'] = None
df_good['connection_pairs'] = None

pkl_files = get_pkl_files(folder_path)

for file in pkl_files:
    df = pd.read_pickle(fr'{folder_path}/{file}')
    mask = df_good['session_id'] == df['session_id'].iloc[0]
    df_good.loc[mask, 'connectivity'] = list(df['connectivity'])
    df_good.loc[mask, 'connection_pairs'] = list(df['connection_pairs'])

In [ ]:
import pandas as pd
base_folder=r"S:\Sachuriga\file_with_table\ripple_ch\Functional_connections"
file=r"63385_2024-07-11_14-42-01_units_table_withDLC.pkl"

df=pd.read_pickle(fr"{base_folder}/{file}")
df[['cell_type','buzaki_cell_type','connectivity','connection_pairs']]
df['connectivity'].iloc[8] = None
df['connectivity'].iloc[18] = None
df['connectivity'].iloc[13] = None
df['connectivity'].iloc[27] = None

df[['cell_type','buzaki_cell_type','connectivity','connection_pairs']]

In [ ]:
df.to_pickle(fr"{base_folder}/{file}")

In [ ]:
import os
import pandas as pd

folder_path = r"S:\Sachuriga\filre_with_table\adjust/test"

# Function to get pickle files
def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

# Define group prefixes
target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']

# Get the list of pickle files
pkl_files = get_pkl_files(folder_path)

# Initialize a list to store all DataFrames
all_dfs = []

# Process each pickle file
for file in pkl_files:
    # Load the DataFrame
    df = pd.read_pickle(os.path.join(folder_path, file))
    # Filter for Pyramidal cells
    df = df[df['cell_type'] == "pyramidal"]
    # Check if the DataFrame is empty after filtering
    if df.empty:
        print(f"Warning: File {file} has no pyramidal cells")
        continue
    # Get the animal_id
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    # Determine the group based on the animal_id prefix
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue  # Skip files that don’t match any group
    # Append the DataFrame to the list
    all_dfs.append(df)

# Concatenate all DataFrames into one
combined_df = pd.concat(all_dfs, ignore_index=True)

# Collect the specified columns collectively for each group
# For control group
control_data = combined_df[combined_df['group_ani'] == 'control'][
    ['addjust y', 'matlab_test_stat_si', 'matlab_sparsity', 'matlab_maxfsize','mean_firing_rate','bursting_index']
]
# For experimental group
exp_data = combined_df[combined_df['group_ani'] == 'exp'][
    ['addjust y', 'matlab_test_stat_si', 'matlab_sparsity', 'matlab_maxfsize','mean_firing_rate','bursting_index']
]

# Optional: If you need the values as lists
control_addjust_y = control_data['addjust y'].to_list()
control_matlab_test_stat_si = control_data['matlab_test_stat_si'].to_list()
control_matlab_sparsity = control_data['matlab_sparsity'].to_list()
control_matlab_maxfsize = control_data['matlab_maxfsize'].to_list()

exp_addjust_y = exp_data['addjust y'].to_list()
exp_matlab_test_stat_si = exp_data['matlab_test_stat_si'].to_list()
exp_matlab_sparsity = exp_data['matlab_sparsity'].to_list()
exp_matlab_maxfsize = exp_data['matlab_maxfsize'].to_list()

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import shapiro, levene, kruskal
import statsmodels.api as sm
from statsmodels.formula.api import ols
import scikit_posthocs as sp

# Folder path
folder_path = r"S:\Sachuriga\file_with_table\ripple_ch"

# Function to get pickle files
def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

# Define group prefixes
target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']

# Get the list of pickle files
pkl_files = get_pkl_files(folder_path)

# Initialize a list to store all DataFrames
all_dfs = []

# Variables to analyze
variables = ['Information_content_rate', 'Sparsity', 'Field_size', 'Averate_rate', 'bursting_index', 'Selectivity']
titles = ['Test Stat SI', 'Sparsity', 'Max Field Size', 'Mean Firing Rate', 'Bursting Index', 'Selectivity']

# Process each pickle file
for file in pkl_files:
    df = pd.read_pickle(os.path.join(folder_path, file))
    #df = df[(df['cell_type'] == "pyramidal") & (df['session'] == "A")]
    df = df[(df['cell_type'] == "pyramidal") & (df['session'] == "A")]
    # df['buzaki_cell_type'] = None
    # for i in range(len(df)):
    #     if df['peak_to_valley'].iloc[i] <= 0.000425:
    #         df['buzaki_cell_type'].iloc[i] = "narrow_spike_interneurons"
    #     elif  (df['peak_to_valley'].iloc[i] > 0.000425) & (df['matlab_acg_tau_rise_1'].iloc[i]> 6):
    #         df['buzaki_cell_type'].iloc[i] = "wide_spike_interneurons"
    #     else:
    #         df['buzaki_cell_type'].iloc[i] = "pyramidal"
    # df = df[df['buzaki_cell_type'] == "pyramidal"]
    
    if df.empty:
        print(f"Warning: File {file} has no data")
        continue
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    # Check if required columns exist
    if not all(var in df.columns for var in variables):
        print(f"Warning: File {file} missing some variables: {set(variables) - set(df.columns)}")
        continue
    # Convert object columns to numeric
    for var in variables:
        df[var] = pd.to_numeric(df[var], errors='coerce')
        if df[var].isna().any():
            print(f"Warning: {var} in {file} contains non-numeric values, converted to NaN")
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue
    all_dfs.append(df)

# Define y-position column
y_pos = "addjust y r2"

# Concatenate all DataFrames into one
combined_df = pd.concat(all_dfs, ignore_index=True)

# Add depth category (deep: >=0, superficial: <0)
combined_df['depth'] = combined_df[y_pos].apply(lambda x: 'deep' if x >= 0 else 'superficial')
combined_df['group_depth'] = combined_df['group_ani'] + "_" + combined_df['depth']

# Verify group_depth labels
print("\nUnique group_depth values:", combined_df['group_depth'].unique())

# Define specific y-limits for each variable
ylim_dict = {
    'Information_content_rate': (0, 4),
    'Sparsity': (0, 1),
    'Field_size': (0, 50),
    'Averate_rate': (0, 8),
    'bursting_index': (0, 5),
    'Selectivity': (0, 40)
}

# Set up the plotting theme
sns.set_theme(style="ticks", palette="pastel")

# Diagnostic tests and plotting for each variable
for var, title in zip(variables, titles):
    print(f"\n--- Analysis for {title} ---")
    
    # Two-Way ANOVA (for reference)
    model = ols(f'{var} ~ C(group_ani) + C(depth) + C(group_ani):C(depth)', data=combined_df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"Two-Way ANOVA for {title} (Reference):")
    print(anova_table)
    
    # Check assumptions
    shapiro_stat, shapiro_p = shapiro(model.resid)
    print(f"\nNormality of residuals (Shapiro-Wilk): stat = {shapiro_stat:.3f}, p = {shapiro_p:.4f}")
    groups = [combined_df[combined_df['group_depth'] == g][var].dropna() for g in combined_df['group_depth'].unique()]
    levene_stat, levene_p = levene(*groups)
    print(f"Homogeneity of variances (Levene): stat = {levene_stat:.3f}, p = {levene_p:.4f}")
    sample_sizes = combined_df['group_depth'].value_counts()
    print("\nSample sizes per group:")
    print(sample_sizes)
    
    # Print sample sizes per group_depth for the variable
    print(f"\nSample sizes per group_depth for {var}:")
    print(combined_df.groupby('group_depth')[var].count())
    
    # Kruskal-Wallis test
    kruskal_stat, kruskal_p = kruskal(*groups)
    print(f"\nKruskal-Wallis Test for {title}:")
    print(f"Stat = {kruskal_stat:.3f}, p = {kruskal_p:.4f}")
    
    # Create figure for boxplot and heatmap
    fig = plt.figure(figsize=(14, 6))  # Increased figure size for clarity
    
    # Boxplot on the left
    ax1 = fig.add_subplot(1, 2, 1)
    sns.boxplot(x='depth', y=var, hue='group_ani', data=combined_df, ax=ax1, 
                palette={'control': 'blue', 'exp': 'cyan'}, width=0.6, dodge=True)
    # sns.stripplot(x='depth', y=var, hue='group_ani', data=combined_df, ax=ax1, 
    #               size=4, jitter=True, dodge=True, 
    #               palette={'control': 'blue', 'exp': 'cyan'}, 
    #               alpha=0.6, edgecolor="gray", linewidth=0.5)
    ax1.set_title(f'{title} by Depth and Group')
    ax1.set_xlabel('Depth')
    ax1.set_ylabel(title)
    ax1.set_ylim(ylim_dict[var])
    handles, labels = ax1.get_legend_handles_labels()
    ax1.legend(handles[:2], ['Control', 'Exp'], title='Group', loc='upper right')
    ax1.legend().set_visible(False)
    
    # Dunn's test and heatmap on the right if significant
    if kruskal_p < 0.05:
        print(f"\nDunn's Post-hoc Test for {title}:")
        # Filter out groups with insufficient data (e.g., <5 samples)
        valid_groups = combined_df.groupby('group_depth').filter(lambda x: len(x[var].dropna()) > 5)['group_depth'].unique()
        combined_df_valid = combined_df[combined_df['group_depth'].isin(valid_groups)]
        
        if len(valid_groups) > 1:  # Proceed only if there are enough groups
            dunn_result = sp.posthoc_dunn(combined_df_valid, val_col=var, group_col='group_depth', p_adjust='bonferroni')
            
            # Ensure unique and sorted group labels
            group_order = sorted(combined_df_valid['group_depth'].unique())
            dunn_result = dunn_result.loc[group_order, group_order]  # Reorder to ensure consistency
            
            # Print Dunn's result for debugging
            print(dunn_result)
            
            ax2 = fig.add_subplot(1, 2, 2)
            
            # Create a mask for the upper triangle (including diagonal)
            mask = np.triu(np.ones_like(dunn_result, dtype=bool))
            
            # Plot heatmap with masked upper triangle
            sns.heatmap(
                dunn_result, 
                annot=True, 
                cmap='hot_r',  # Reversed 'hot' colormap for better contrast
                vmin=0, 
                vmax=0.1, 
                fmt='.3f', 
                linewidths=0.5, 
                ax=ax2,
                mask=mask,  # Apply the mask
                cbar_kws={'label': 'p-value'},
                square=True  # Ensure square cells
            )
            
            ax2.set_title(f"Dunn's Post-hoc Test P-values for {title}")
            
            # Rotate x-axis labels for better readability
            ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
            ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)
        else:
            ax2 = fig.add_subplot(1, 2, 2)
            ax2.text(0.5, 0.5, 'Insufficient data for Dunn’s test\n(<2 groups with enough samples)', 
                     ha='center', va='center', fontsize=12)
            ax2.set_title(f"Dunn's Post-hoc Test for {title}")
            ax2.axis('off')
    else:
        # If not significant, add a text box instead of heatmap
        ax2 = fig.add_subplot(1, 2, 2)
        ax2.text(0.5, 0.5, 'No significant differences\n(Kruskal-Wallis p > 0.05)', 
                 ha='center', va='center', fontsize=12)
        ax2.set_title(f"Dunn's Post-hoc Test for {title}")
        ax2.axis('off')
    
    sns.despine(ax=ax1, offset=10, trim=True)
    plt.tight_layout(pad=2.0)  # Add padding for layout
    plt.show()

In [ ]:
combined_df

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

# Data loading
folder_path = r"S:\Sachuriga\filre_with_table\adjust_y_with _meanVAlue\clusters_with_tsneLabel\ripple_max\ripple_py"

def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']
pkl_files = get_pkl_files(folder_path)

all_dfs = []
for file in pkl_files:
    df = pd.read_pickle(os.path.join(folder_path, file))
    #df = df[(df['cell_type'] == "pyramidal") & (df['session'] == "A")]
    df = df[df['session'] == "A"]
    df['buzaki_cell_type'] = None
    for i in range(len(df)):
        if df['peak_to_valley'].iloc[i] <= 0.000425:
            df['buzaki_cell_type'].iloc[i] = "narrow_spike_interneurons"
        elif  (df['peak_to_valley'].iloc[i] > 0.000425) & (df['matlab_acg_tau_rise_1'].iloc[i]> 6):
            df['buzaki_cell_type'].iloc[i] = "wide_spike_interneurons"
        else:
            df['buzaki_cell_type'].iloc[i] = "pyramidal"
    df = df[(df['buzaki_cell_type'] == "pyramidal")]

    if df.empty:
        print(f"Warning: File {file} has no pyramidal cells")
        continue
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue
    all_dfs.append(df)


y_pos = "addjust y r2"
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df['depth'] = combined_df[y_pos].apply(lambda x: 'deep' if x > 0 else 'superficial')
combined_df['group_depth'] = combined_df['group_ani'] + '_' + combined_df['depth']

# Variables and titles
variables = ['matlab_test_stat_si', 'matlab_sparsity', 'matlab_maxfsize', 'mean_firing_rate', 'bursting_index']
titles = ['Test Stat SI', 'Sparsity', 'Max F Size', 'Mean Firing Rate', 'Bursting Index']

# ECDF Plotting with 2x5 grid (2 rows for depth, 5 columns for variables)
sns.set_theme(style="ticks")
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# Custom palette: control in blue, exp in cyan
palette = {'control': 'blue', 'exp': 'cyan'}

for i, (var, title) in enumerate(zip(variables, titles)):
    # Control_deep vs. Exp_deep
    control_deep = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'deep')][var].dropna()
    exp_deep = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'deep')][var].dropna()
    ks_stat_deep, p_valued = ks_2samp(control_deep, exp_deep, nan_policy='omit')
    print(f"{title} - Control_deep vs. Exp_deep: statistic={ks_stat_deep:.3f}, p-value={p_valued:.4f}")
    
    # Control_superficial vs. Exp_superficial
    control_superficial = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'superficial')][var].dropna()
    exp_superficial = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'superficial')][var].dropna()
    ks_stat_sup, p_values = ks_2samp(control_superficial, exp_superficial, nan_policy='omit')
    print(f"{title} - Control_superficial vs. Exp_superficial: statistic={ks_stat_sup:.3f}, p-value={p_values:.4f}")

    # Deep subplot
    sns.ecdfplot(data=combined_df[combined_df['depth'] == 'deep'], x=var, hue='group_ani', 
                ax=axes[0, i], palette=palette)
    axes[0, i].set_title(f'CDF of {title} (Deep)', fontsize=12, pad=10)
    axes[0, i].set_xlabel(title, fontsize=10)
    axes[0, i].set_ylabel('Cumulative Probability', fontsize=10)
    axes[0, i].legend(title='Group', labels=['Control', 'Exp'], 
                    handles=[plt.Line2D([0], [0], color=palette['control'], lw=2),
                            plt.Line2D([0], [0], color=palette['exp'], lw=2)], 
                    loc='lower right', fontsize=8, title_fontsize=10)
    axes[0, i].text(0.5, 0.1, f'ks2 test, p = {p_valued:.4f}', ha='center', va='bottom', 
                    transform=axes[0, i].transAxes, fontsize=10, color='black')

    # Superficial subplot
    sns.ecdfplot(data=combined_df[combined_df['depth'] == 'superficial'], x=var, hue='group_ani', 
                ax=axes[1, i], palette=palette)
    axes[1, i].set_title(f'CDF of {title} (Superficial)', fontsize=12, pad=10)
    axes[1, i].set_xlabel(title, fontsize=10)
    axes[1, i].set_ylabel('Cumulative Probability', fontsize=10)
    axes[1, i].legend(title='Group', labels=['Control', 'Exp'], 
                    handles=[plt.Line2D([0], [0], color=palette['control'], lw=2),
                            plt.Line2D([0], [0], color=palette['exp'], lw=2)], 
                    loc='lower right', fontsize=8, title_fontsize=10)
    axes[1, i].text(0.5, 0.1, f'ks2 test, p = {p_values:.4f}', ha='center', va='bottom', 
                    transform=axes[1, i].transAxes, fontsize=10, color='black')
sns.despine(offset=10, trim=True)
plt.tight_layout()
plt.show()

In [ ]:
pip uninstall scipy

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np

# Data loading
folder_path = r"S:\Sachuriga\filre_with_table\adjust_y_with _meanVAlue\clusters_with_tsneLabel\ripple_max\ripple_py\uniques\ripple_max"

def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']
pkl_files = get_pkl_files(folder_path)

# Variables to analyze
variables = ['Information_content_rate', 'Sparsity', 'Field_size', 'Averate_rate', 'bursting_index', 'Selectivity']
titles = ['Information_content_rate', 'Sparsity', 'Max Field Size', 'Mean Firing Rate', 'Bursting Index', 'Selectivity']
all_dfs = []

# Process each pickle file
for file in pkl_files:
    try:
        df = pd.read_pickle(os.path.join(folder_path, file))
    except Exception as e:
        continue
    #df = df[df['session'] == "A"]
    df['buzaki_cell_type'] = None
    for i in range(len(df)):
        if df['peak_to_valley'].iloc[i] <= 0.000425:
            df['buzaki_cell_type'].iloc[i] = "narrow_spike_interneurons"
        elif  (df['peak_to_valley'].iloc[i] > 0.000425) & (df['matlab_acg_tau_rise_1'].iloc[i]> 6):
            df['buzaki_cell_type'].iloc[i] = "wide_spike_interneurons"
        else:
            df['buzaki_cell_type'].iloc[i] = "pyramidal"
    #df = df[(df['buzaki_cell_type'] == "pyramidal") & (df['cell_type'] == "pyramidal")]
    df = df[df['cell_type'] == "pyramidal"]
    if df.empty:
        print(f"Warning: File {file} has no pyramidal cells")
        continue
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue
    all_dfs.append(df)

y_pos = "addjust y r2"
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df['depth'] = combined_df[y_pos].apply(lambda x: 'deep' if x > 0 else 'superficial')
combined_df['group_depth'] = combined_df['group_ani'] + '_' + combined_df['depth']

# ECDF Plotting with 2x6 grid (2 rows for depth, 6 columns for variables)
#sns.set_theme(style="ticks")
fig, axes = plt.subplots(2, 6, figsize=(20, 6))

# Custom palette: control in blue, exp in cyan
palette = {'control': 'blue', 'exp': 'cyan'}

for i, (var, title) in enumerate(zip(variables, titles)):
    # Control_deep vs. Exp_deep
    control_deep = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'deep')][var].dropna()
    exp_deep = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'deep')][var].dropna()
    ks_stat_deep, p_valued = ks_2samp(control_deep, exp_deep, nan_policy='omit')
    print(f"{title} - Control_deep vs. Exp_deep: statistic={ks_stat_deep:.3f}, p-value={p_valued:.4f}")
    
    # Control_superficial vs. Exp_superficial
    control_superficial = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'superficial')][var].dropna()
    exp_superficial = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'superficial')][var].dropna()
    ks_stat_sup, p_values = ks_2samp(control_superficial, exp_superficial, nan_policy='omit')
    print(f"{title} - Control_superficial vs. Exp_superficial: statistic={ks_stat_sup:.3f}, p-value={p_values:.4f}")

    # Deep subplot
    sns.ecdfplot(data=combined_df[combined_df['depth'] == 'deep'], x=var, hue='group_ani', 
                 ax=axes[0, i], palette=palette)
    axes[0, i].set_title(f'{title} (Deep)', fontsize=12, pad=10)
    axes[0, i].set_xlabel(title, fontsize=10)
    axes[0, i].set_ylabel('Cumulative Probability', fontsize=10)
    axes[0, i].legend(title='Group', labels=['Control', 'Exp'], 
                      handles=[plt.Line2D([0], [0], color=palette['control'], lw=2),
                               plt.Line2D([0], [0], color=palette['exp'], lw=2)], 
                      loc='lower right', fontsize=8, title_fontsize=10)
    axes[0, i].text(0.7, 0.1, f'ks2 , p = {p_valued:.4f}', ha='center', va='bottom', 
                    transform=axes[0, i].transAxes, fontsize=10, color='black')
    axes[0, i].legend().set_visible(False)
    axes[0, i].spines['top'].set_visible(False)
    axes[0, i].spines['right'].set_visible(False)

    # Add inset bar plot for deep
    inset_ax_deep = inset_axes(axes[0, i], width="100%", height="100%", loc='lower right',
                     bbox_to_anchor=(0.7, 0.3, 0.3, 0.3),  # 4-tuple: x0, y0, width, height
                     bbox_transform= axes[0, i].transAxes)
    
    means_deep = [control_deep.mean(), exp_deep.mean()]
    sems_deep = [control_deep.sem(), exp_deep.sem()]
    inset_ax_deep.bar([.2, .8], means_deep, yerr=sems_deep, color=[palette['control'], palette['exp']], 
                      capsize=3, width=0.45)
    inset_ax_deep.set_xticks([.2, .8])
    inset_ax_deep.set_xticklabels(['CRs +', 'CRs -'], fontsize=6)
    inset_ax_deep.set_yticks([])  # Hide y-axis ticks for simplicity
    inset_ax_deep.spines['top'].set_visible(False)
    inset_ax_deep.spines['right'].set_visible(False)
    #inset_ax_deep.set_title('Mean ± SEM', fontsize=8)

    # Superficial subplot
    sns.ecdfplot(data=combined_df[combined_df['depth'] == 'superficial'], x=var, hue='group_ani', 
                 ax=axes[1, i], palette=palette)
    axes[1, i].set_title(f'{title} (Superficial)', fontsize=12, pad=10)
    axes[1, i].set_xlabel(title, fontsize=10)
    axes[1, i].set_ylabel('Cumulative Probability', fontsize=10)
    axes[1, i].legend(title='Group', labels=['Control', 'Exp'], 
                      handles=[plt.Line2D([0], [0], color=palette['control'], lw=2),
                               plt.Line2D([0], [0], color=palette['exp'], lw=2)], 
                      loc='lower right', fontsize=8, title_fontsize=10)
    axes[1, i].text(0.7, 0.1, f'ks2 , p = {p_values:.4f}', ha='center', va='bottom', 
                    transform=axes[1, i].transAxes, fontsize=10, color='black')
    axes[1, i].legend().set_visible(False)
    axes[1, i].spines['top'].set_visible(False)
    axes[1, i].spines['right'].set_visible(False)

    # Add inset bar plot for superficial
    inset_ax_sup = inset_axes(axes[1, i], width="100%", height="100%", loc='lower right',
                     bbox_to_anchor=(0.7, 0.3, 0.3, 0.3),  # 4-tuple: x0, y0, width, height
                     bbox_transform= axes[1, i].transAxes)
    
    
    means_sup = [control_superficial.mean(), exp_superficial.mean()]
    sems_sup = [control_superficial.sem(), exp_superficial.sem()]
    inset_ax_sup.bar([.2, .8], means_sup, yerr=sems_sup, color=[palette['control'], palette['exp']], 
                     capsize=3, width=0.45)
    inset_ax_sup.set_xticks([.2, .8])
    inset_ax_sup.set_xticklabels(['CRs +', 'CRs -'], fontsize=6)
    inset_ax_sup.set_yticks([])  # Hide y-axis ticks for simplicity
    inset_ax_sup.spines['top'].set_visible(False)
    inset_ax_sup.spines['right'].set_visible(False)
    #inset_ax_sup.set_title('Mean ± SEM', fontsize=8)

#sns.despine(offset=10, trim=True)
plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, shapiro, ttest_ind, mannwhitneyu
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np

# Data loading
folder_path = r"S:\Sachuriga\file_with_table\ripple_ch"

def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']
pkl_files = get_pkl_files(folder_path)

# Variables to analyze
variables = ['Information_content_rate', 'Sparsity', 'Field_size', 'Averate_rate', 'bursting_index', 'Selectivity']
titles = ['Information_content_rate', 'Sparsity', 'Max Field Size', 'Mean Firing Rate', 'Bursting Index', 'Selectivity']
all_dfs = []

# Process each pickle file
for file in pkl_files:
    try:
        df = pd.read_pickle(os.path.join(folder_path, file))
    except Exception as e:
        continue
    #df = df[(df['cell_type'] == "pyramidal") & (df['session'] == "A")]

    df = df[(df['buzaki_py_cell_type'] == "pyramidal") & (df['session'] == "A")]
    #df = df[df['session'] == "A"]
    if df.empty:
        print(f"Warning: File {file} has no pyramidal cells")
        continue
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue
    all_dfs.append(df)

y_pos = "addjust y r2"
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df['depth'] = combined_df[y_pos].apply(lambda x: 'deep' if x > 0 else 'superficial')
combined_df['group_depth'] = combined_df['group_ani'] + '_' + combined_df['depth']

# Convert variables to numeric and handle invalid values
for var in variables:
    combined_df[var] = pd.to_numeric(combined_df[var], errors='coerce')
    if combined_df[var].isna().any():
        print(f"Warning: {var} contains NaN values after conversion to numeric")

# Function to test normality and choose test
def choose_stat_test(data1, data2, var_name, group1_name, group2_name):
    # Drop NaN values and ensure numeric
    data1 = data1.dropna()
    data2 = data2.dropna()
    
    # Check for non-numeric values
    if not np.issubdtype(data1.dtype, np.number) or not np.issubdtype(data2.dtype, np.number):
        print(f"Error: Non-numeric data detected in {var_name} for {group1_name} or {group2_name}")
        print(f"{group1_name} dtype: {data1.dtype}, {group2_name} dtype: {data2.dtype}")
        print(f"{group1_name} sample: {data1.head()}")
        print(f"{group2_name} sample: {data2.head()}")
        return "Invalid", np.nan, np.nan

    # Check if data is empty after dropping NaNs
    if len(data1) == 0 or len(data2) == 0:
        print(f"Error: Empty dataset for {var_name} in {group1_name} or {group2_name} after dropping NaNs")
        return "Empty", np.nan, np.nan

    # Perform Shapiro-Wilk test for normality
    stat1, p1 = shapiro(data1)
    stat2, p2 = shapiro(data2)
    
    print(f"{var_name} - {group1_name} Shapiro-Wilk: p={p1:.4f}")
    print(f"{var_name} - {group2_name} Shapiro-Wilk: p={p2:.4f}")
    
    # Choose test based on normality
    if p1 > 0.05 and p2 > 0.05:  # Both are normal
        print(f"{var_name} - Using t-test (both groups normal)")
        stat, p = ttest_ind(data1, data2, equal_var=True)  # Assuming equal variances
        test_name = "t-test"
    else:
        print(f"{var_name} - Using Mann-Whitney U test (non-normal distribution)")
        stat, p = mannwhitneyu(data1, data2, alternative='two-sided')
        test_name = "Mann-Whitney U"
    
    return test_name, stat, p

# ECDF Plotting with 2x6 grid
fig, axes = plt.subplots(2, 6, figsize=(20, 6))
palette = {'control': 'blue', 'exp': 'cyan'}

for i, (var, title) in enumerate(zip(variables, titles)):
    # Control_deep vs. Exp_deep
    control_deep = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'deep')][var]
    exp_deep = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'deep')][var]
    
    # Choose test for deep
    test_name_deep, stat_deep, p_valued = choose_stat_test(control_deep, exp_deep, title, "Control_deep", "Exp_deep")
    print(f"{title} - Control_deep vs. Exp_deep ({test_name_deep}): statistic={stat_deep:.3f}, p-value={p_valued:.4f}")
    
    # Control_superficial vs. Exp_superficial
    control_superficial = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'superficial')][var]
    exp_superficial = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'superficial')][var]
    
    # Choose test for superficial
    test_name_sup, stat_sup, p_values = choose_stat_test(control_superficial, exp_superficial, title, "Control_superficial", "Exp_superficial")
    print(f"{title} - Control_superficial vs. Exp_superficial ({test_name_sup}): statistic={stat_sup:.3f}, p-value={p_values:.4f}")

    ks_stat_deep, p_value_d = ks_2samp(control_deep, exp_deep, nan_policy='omit')
    ks_stat_sup, p_values_s = ks_2samp(control_superficial, exp_superficial, nan_policy='omit')

    
    # Skip plotting if test is invalid or empty
    if test_name_deep in ["Invalid", "Empty"] or test_name_sup in ["Invalid", "Empty"]:
        print(f"Skipping plot for {title} due to invalid or empty data")
        continue

    # Deep subplot
    sns.ecdfplot(data=combined_df[combined_df['depth'] == 'deep'], x=var, hue='group_ani', 
                 ax=axes[0, i], palette=palette)
    axes[0, i].set_title(f'{title} (Deep)', fontsize=12, pad=10)
    axes[0, i].set_xlabel(title, fontsize=10)
    axes[0, i].set_ylabel('Cumulative Probability', fontsize=10)
    axes[0, i].legend(title='Group', labels=['Control', 'Exp'], 
                      handles=[plt.Line2D([0], [0], color=palette['control'], lw=2),
                               plt.Line2D([0], [0], color=palette['exp'], lw=2)], 
                      loc='lower right', fontsize=8, title_fontsize=10)
    axes[0, i].text(0.7, 0.1, f'U test, Adj p = {p_valued:.4f}', ha='center', va='bottom', 
                    transform=axes[0, i].transAxes, fontsize=10, color='black')
    axes[0, i].text(0.7, 0.01, f'ks2 , p = {p_value_d:.4f}', ha='center', va='bottom', 
            transform=axes[0, i].transAxes, fontsize=10, color='black')


    axes[0, i].legend().set_visible(False)
    axes[0, i].spines['top'].set_visible(False)
    axes[0, i].spines['right'].set_visible(False)

    # Inset bar plot for deep
    inset_ax_deep = inset_axes(axes[0, i], width="100%", height="100%", loc='lower right',
                               bbox_to_anchor=(0.7, 0.3, 0.3, 0.3), bbox_transform=axes[0, i].transAxes)
    means_deep = [control_deep.mean(), exp_deep.mean()]
    sems_deep = [control_deep.sem(), exp_deep.sem()]
    inset_ax_deep.bar([.2, .8], means_deep, yerr=sems_deep, color=[palette['control'], palette['exp']], 
                      capsize=3, width=0.45)
    inset_ax_deep.set_xticks([.2, .8])
    inset_ax_deep.set_xticklabels(['CRs +', 'CRs -'], fontsize=6)
    inset_ax_deep.set_yticks([])
    inset_ax_deep.spines['top'].set_visible(False)
    inset_ax_deep.spines['right'].set_visible(False)

    # Superficial subplot
    sns.ecdfplot(data=combined_df[combined_df['depth'] == 'superficial'], x=var, hue='group_ani', 
                 ax=axes[1, i], palette=palette)
    axes[1, i].set_title(f'{title} (Superficial)', fontsize=12, pad=10)
    axes[1, i].set_xlabel(title, fontsize=10)
    axes[1, i].set_ylabel('Cumulative Probability', fontsize=10)
    axes[1, i].legend(title='Group', labels=['Control', 'Exp'], 
                      handles=[plt.Line2D([0], [0], color=palette['control'], lw=2),
                               plt.Line2D([0], [0], color=palette['exp'], lw=2)], 
                      loc='lower right', fontsize=8, title_fontsize=10)
    axes[1, i].text(0.7, 0.1, f'U test,Adj p = {p_values:.4f}', ha='center', va='bottom', 
                    transform=axes[1, i].transAxes, fontsize=10, color='black')
    axes[1, i].text(0.7, 0.01, f'ks2 , p = {p_values_s:.4f}', ha='center', va='bottom', 
                transform=axes[1, i].transAxes, fontsize=10, color='black')
    axes[1, i].legend().set_visible(False)
    axes[1, i].spines['top'].set_visible(False)
    axes[1, i].spines['right'].set_visible(False)

    # Inset bar plot for superficial
    inset_ax_sup = inset_axes(axes[1, i], width="100%", height="100%", loc='lower right',
                              bbox_to_anchor=(0.7, 0.3, 0.3, 0.3), bbox_transform=axes[1, i].transAxes)
    means_sup = [control_superficial.mean(), exp_superficial.mean()]
    sems_sup = [control_superficial.sem(), exp_superficial.sem()]
    inset_ax_sup.bar([.2, .8], means_sup, yerr=sems_sup, color=[palette['control'], palette['exp']], 
                     capsize=3, width=0.45)
    inset_ax_sup.set_xticks([.2, .8])
    inset_ax_sup.set_xticklabels(['CRs +', 'CRs -'], fontsize=6)
    inset_ax_sup.set_yticks([])
    inset_ax_sup.spines['top'].set_visible(False)
    inset_ax_sup.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, shapiro, ttest_ind, mannwhitneyu
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np

# Data loading
folder_path = r"S:\Sachuriga\file_with_table\ripple_ch"

def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']
pkl_files = get_pkl_files(folder_path)

# Variables to analyze
variables = ['Information_content_rate', 'Sparsity', 'Field_size', 'Averate_rate', 'bursting_index', 'Selectivity']
titles = ['Information_content_rate', 'Sparsity', 'Max Field Size', 'Mean Firing Rate', 'Bursting Index', 'Selectivity']

metrics_labels = ['Information content rate (spikes/bit)', 'Sparsity',  'Field size (percentage)', 'Averate firing rate (Hz)', 'Bursting Index', 'Selectivity (max rate/ mean rate)']

all_dfs = []

# Process each pickle file
for file in pkl_files:
    try:
        df = pd.read_pickle(os.path.join(folder_path, file))
    except Exception as e:
        continue
    df['buzaki_cell_type'] = None
    for i in range(len(df)):
        if df['peak_to_valley'].iloc[i] <= 0.000425:
            df['buzaki_cell_type'].iloc[i] = "narrow_spike_interneurons"
        elif (df['peak_to_valley'].iloc[i] > 0.000425) & (df['matlab_acg_tau_rise_1'].iloc[i] > 6):
            df['buzaki_cell_type'].iloc[i] = "wide_spike_interneurons"
        else:
            df['buzaki_cell_type'].iloc[i] = "pyramidal"
    df = df[(df['cell_type'] == "pyramidal") & (df['session'] == "A")]
    if df.empty:
        print(f"Warning: File {file} has no pyramidal cells")
        continue
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue
    all_dfs.append(df)

y_pos = "addjust y r2"
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df['depth'] = combined_df[y_pos].apply(lambda x: 'deep' if x > 0 else 'superficial')
combined_df['group_depth'] = combined_df['group_ani'] + '_' + combined_df['depth']

# Convert variables to numeric and handle invalid values
for var in variables:
    combined_df[var] = pd.to_numeric(combined_df[var], errors='coerce')
    if combined_df[var].isna().any():
        print(f"Warning: {var} contains NaN values after conversion to numeric")

# Function to test normality and choose test
def choose_stat_test(data1, data2, var_name, group1_name, group2_name):
    data1 = data1.dropna()
    data2 = data2.dropna()
    
    if not np.issubdtype(data1.dtype, np.number) or not np.issubdtype(data2.dtype, np.number):
        print(f"Error: Non-numeric data detected in {var_name} for {group1_name} or {group2_name}")
        return "Invalid", np.nan, np.nan

    if len(data1) == 0 or len(data2) == 0:
        print(f"Error: Empty dataset for {var_name} in {group1_name} or {group2_name} after dropping NaNs")
        return "Empty", np.nan, np.nan

    stat1, p1 = shapiro(data1)
    stat2, p2 = shapiro(data2)
    
    print(f"{var_name} - {group1_name} Shapiro-Wilk: p={p1:.4f}")
    print(f"{var_name} - {group2_name} Shapiro-Wilk: p={p2:.4f}")
    
    if p1 > 0.05 and p2 > 0.05:
        print(f"{var_name} - Using t-test (both groups normal)")
        stat, p = ttest_ind(data1, data2, equal_var=True)
        test_name = "t-test"
    else:
        print(f"{var_name} - Using Mann-Whitney U test (non-normal distribution)")
        stat, p = mannwhitneyu(data1, data2, alternative='two-sided')
        test_name = "Mann-Whitney U"
    
    return test_name, stat, p

# Bar Plotting with 2x6 grid
fig, axes = plt.subplots(2, 6, figsize=(20, 9))
palette = {'control': '#2b4d5e', 'exp': '#BC554E'}

for i, (var, title) in enumerate(zip(variables, titles)):
    # Control_deep vs. Exp_deep
    title = metrics_labels[i] 
    control_deep = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'deep')][var]
    exp_deep = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'deep')][var]
    
    test_name_deep, stat_deep, p_valued = choose_stat_test(control_deep, exp_deep, title, "Control_deep", "Exp_deep")
    print(f"{title} - Control_deep vs. Exp_deep ({test_name_deep}): statistic={stat_deep:.3f}, p-value={p_valued:.4f}")
    
    # Control_superficial vs. Exp_superficial
    control_superficial = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'superficial')][var]
    exp_superficial = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'superficial')][var]
    
    test_name_sup, stat_sup, p_values = choose_stat_test(control_superficial, exp_superficial, title, "Control_superficial", "Exp_superficial")
    print(f"{title} - Control_superficial vs. Exp_superficial ({test_name_sup}): statistic={stat_sup:.3f}, p-value={p_values:.4f}")

    ks_stat_deep, p_value_d = ks_2samp(control_deep, exp_deep, nan_policy='omit')
    ks_stat_sup, p_values_s = ks_2samp(control_superficial, exp_superficial, nan_policy='omit')

    if test_name_deep in ["Invalid", "Empty"] or test_name_sup in ["Invalid", "Empty"]:
        print(f"Skipping plot for {title} due to invalid or empty data")
        continue

    # Deep subplot - Bar plot
    sns.barplot(data=combined_df[combined_df['depth'] == 'deep'], x='group_ani', y=var, 
                ax=axes[0, i], palette=palette, order=['control', 'exp'], capsize=0.1)
    sns.stripplot(data=combined_df[combined_df['depth'] == 'deep'], x='group_ani', y=var, 
                  ax=axes[0, i], color='#ECE0CA', size=3, order=['control', 'exp'], jitter=0.2)
    
    #axes[0, i].set_title(f'{title} (Deep)', fontsize=12, pad=10)
    axes[0, i].set_xlabel('', fontsize=10)
    axes[0, i].set_ylabel(title, fontsize=16)
    axes[0, i].set_xticks([0, 1])
    axes[0, i].set_xticklabels(['CRs +', 'CRs -'], fontsize=8)
    axes[0, i].text(0.5, 0.9, f'p = {p_valued:.4f}', ha='center', va='top', 
                    transform=axes[0, i].transAxes, fontsize=16, color='black')
    # axes[0, i].text(0.5, 0.8, f'KS p = {p_value_d:.4f}', ha='center', va='top', 
    #                 transform=axes[0, i].transAxes, fontsize=10, color='black')
    if title=='Bursting Index':
        axes[0, i].set_ylim([0,5])
    axes[0, i].spines['top'].set_visible(False)
    axes[0, i].spines['right'].set_visible(False)
    axes[0, i].tick_params(axis='y', labelsize=16)
    axes[0, i].tick_params(axis='x', labelsize=16)
    # Superficial subplot - Bar plot
    sns.barplot(data=combined_df[combined_df['depth'] == 'superficial'], x='group_ani', y=var, 
                ax=axes[1, i], palette=palette, order=['control', 'exp'], capsize=0.1)
    sns.stripplot(data=combined_df[combined_df['depth'] == 'superficial'], x='group_ani', y=var, 
                  ax=axes[1, i], color='#ECE0CA', size=3, order=['control', 'exp'], jitter=0.2)
    
    #axes[1, i].set_title(f'{title} (Superficial)', fontsize=12, pad=10)
    if title=='Bursting Index':
        axes[1, i].set_ylim([0,5])
    
    axes[1, i].set_xlabel('', fontsize=10)
    axes[1, i].set_ylabel(title, fontsize=16)
    axes[1, i].set_xticks([0, 1])
    axes[1, i].set_xticklabels(['CRs +', 'CRs -'], fontsize=8)
    axes[1, i].text(0.5, 0.9, f'p = {p_values:.4f}', ha='center', va='top', 
                    transform=axes[1, i].transAxes, fontsize=16, color='black')
    # axes[1, i].text(0.5, 0.8, f'KS p = {p_values_s:.4f}', ha='center', va='top', 
    #                 transform=axes[1, i].transAxes, fontsize=10, color='black')
    axes[1, i].spines['top'].set_visible(False)
    axes[1, i].spines['right'].set_visible(False)
    axes[1, i].tick_params(axis='y', labelsize=16)
    axes[1, i].tick_params(axis='x', labelsize=16)


fig.patch.set_facecolor('none')
# Save the plot with a transparent background
plt.tight_layout()
plt.savefig(r'Q:\sachuriga\CR_CA1_paper\Results\DeepVSsuperficial/barplot_only.eps', transparent=True, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

folder_path = r"S:\Sachuriga\file_with_table\ripple_ch"

def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']

pkl_files = get_pkl_files(folder_path)
all_dfs = []
y_pos = 'addjust y r2'

for file in pkl_files:
    df = pd.read_pickle(os.path.join(folder_path, file))
    df = df[df['buzaki_py_cell_type'] == "pyramidal"]
    #df = df[(df['cell_type'] == "pyramidal") & (df['session'] == "A")]
    if df.empty:
        print(f"Warning: File {file} has no pyramidal cells")
        continue
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue
    all_dfs.append(df)
combined_df = pd.concat(all_dfs, ignore_index=True)

y_pos = "addjust y r2"
combined_df['depth'] = combined_df[y_pos].apply(lambda x: 'deep' if x > 0 else 'superficial')
combined_df['group_depth'] = combined_df['group_ani'] + '_' + combined_df['depth']

# Separate data for KS test and quantiles
control_data = combined_df[combined_df['group_ani'] == 'control'][y_pos].dropna()
exp_data = combined_df[combined_df['group_ani'] == 'exp'][y_pos].dropna()

# Perform KS test
ks_stat, ks_p = ks_2samp(control_data, exp_data)
print(f"KS Test Results:")
print(f"Statistic: {ks_stat:.4f}")
print(f"p-value: {ks_p:.4f}")

# Calculate 2.5th and 97.5th percentiles
control_q2_5 = np.percentile(control_data, 5)
control_q97_5 = np.percentile(control_data, 95)
exp_q2_5 = np.percentile(exp_data, 5)
exp_q97_5 = np.percentile(exp_data, 95)
print(f"\nQuantiles:")
print(f"Control: 2.5% = {control_q2_5:.2f}, 97.5% = {control_q97_5:.2f}")
print(f"Experimental: 2.5% = {exp_q2_5:.2f}, 97.5% = {exp_q97_5:.2f}")

# Create figure with three subplots: Histogram, CDF, and Barplot
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 6), dpi=100)
sns.set(style="white")

# Define bins for the histogram (e.g., 50 bins across the data range)
data_range = combined_df[y_pos].dropna()
bins = np.linspace(data_range.min(), data_range.max(), 51)  # 50 bins

# First histogram: control and exp groups (filled)
sns.histplot(data=combined_df, x=y_pos, hue='group_ani', 
             element="step", stat="count", bins=bins,
             palette=['blue', 'cyan'], alpha=1, ax=ax1)

# Filter for experimental group only
exp_df = combined_df[combined_df['group_ani'] == 'exp']

# Second histogram: exp group with edge line only
sns.histplot(data=exp_df, x=y_pos, 
             element="step", stat="count", fill=False,
             color='cyan', linewidth=2, bins=bins, ax=ax1, alpha=0.8)

ax1.set_title('Histogram of soma position relative to the ripple peak')
ax1.set_xlabel('Soma position relative to the ripple peak')
ax1.set_ylabel('Count')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(labels=['CRs +/-', 'CRs -/-'])

# CDF (middle subplot)
sns.ecdfplot(data=combined_df, x=y_pos, hue='group_ani', 
             palette=['blue', 'cyan'], ax=ax2)
ax2.set_title('Cumulative Distribution of soma position relative to the ripple peak')
ax2.set_xlabel('Soma position relative to the ripple peak')
ax2.set_ylabel('Cumulative Proportion')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Add 2.5th and 97.5th percentile lines for control
ax2.axvline(x=control_q2_5, color='blue', linestyle='--', alpha=0.7,
            label=f'CRs +/- 2.5% ({control_q2_5:.2f})')
ax2.axvline(x=control_q97_5, color='blue', linestyle='--', alpha=0.7,
            label=f'CRs +/- 97.5% ({control_q97_5:.2f})')

# Add 2.5th and 97.5th percentile lines for experimental
ax2.axvline(x=exp_q2_5, color='cyan', linestyle='--', alpha=0.7,
            label=f'CRs -/- 2.5% ({exp_q2_5:.2f})')
ax2.axvline(x=exp_q97_5, color='cyan', linestyle='--', alpha=0.7,
            label=f'CRs -/- 97.5% ({exp_q97_5:.2f})')

ax2.legend()
# Create inset axes for bar plot in top-right corner
inset_ax = inset_axes(ax2, width="90%", height="90%", loc='lower right',
                    bbox_to_anchor=(0.7, 0.4, 0.3, 0.3),  # 4-tuple: x0, y0, width, height
                    bbox_transform=ax2.transAxes)

counts = combined_df.groupby(['group_ani', 'depth']).size().unstack(fill_value=0)
counts['total'] = counts['deep'] + counts['superficial']
counts['deep_ratio'] = counts['deep'] / counts['total'] * 100
counts['superficial_ratio'] = counts['superficial'] / counts['total'] * 100

# Prepare data for the bar plot
groups = counts.index
deep_ratios = counts['deep_ratio']
superficial_ratios = counts['superficial_ratio']

# Define colors for each group
colors = {'control': 'blue', 'exp': 'cyan'}  # Colors inspired by the image
# Plot filled bars for deep
x=[0.2,0.8]
inset_ax.bar(x, deep_ratios, 0.8, label='Deep', color=[colors[g] for g in groups], alpha=0.6)

# Plot outline bars for superficial (stacked on top of deep)
inset_ax.bar(x, superficial_ratios, 0.8, bottom=deep_ratios, 
        edgecolor=[colors[g] for g in groups], linewidth=2, fill=False, label='Superficial')

# Add percentage labels
for i, group in enumerate(groups):
    # Deep percentage
    deep_height = deep_ratios[i]
    inset_ax.text(i, deep_height/2, f'{deep_height:.0f}%', ha='center', va='center', fontsize=12, fontweight='bold')
    
    # Superficial percentage
    superficial_height = superficial_ratios[i]
    total_height = deep_height + superficial_height
    inset_ax.text(i, deep_height + superficial_height/2, f'{superficial_height:.0f}%', 
             ha='center', va='center', fontsize=12, fontweight='bold')

# Customize the plot
inset_ax.set_xticks(x)
inset_ax.set_xticklabels(['CRs +','CRs -'])
inset_ax.set_ylim(0, 100)  # Since we're plotting percentages
#inset_ax.set_title('Proportion of Deep vs Superficial by Group')
#inset_ax.set_xlabel('Group')
#inset_ax.set_ylabel('Percentage (%)')
#inset_ax.set_yticklabels().set_visible(False)
inset_ax.tick_params(axis='y', which='both', length=0, labelleft=False)
inset_ax.spines['top'].set_visible(False)
inset_ax.spines['right'].set_visible(False)
inset_ax.spines['left'].set_visible(False)


# Add KS test results as text
ks_text = f"KS Test:\nStat = {ks_stat:.4f}\np = {ks_p:.4f}"
ax2.text(0.05, 0.95, ks_text, transform=ax2.transAxes, fontsize=10,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
# Barplot (right subplot)
palette = {
    'control_deep': 'blue',
    'control_superficial': 'blue',
    'exp_deep': 'cyan',
    'exp_superficial': 'cyan'
}

# Create barplot with adjusted width to reduce distance
sns.barplot(data=combined_df, x='group_ani', y=y_pos, hue='group_depth', 
            palette=palette, ax=ax3, width=0.9)

# Customize bars for superficial layers (no face color, only edge)
for patch in ax3.patches:
    hue = ax3.get_legend().get_texts()[ax3.patches.index(patch) % len(ax3.get_legend().get_texts())].get_text()
    if 'superficial' in hue:
        patch.set_facecolor('none')  # Remove face color
        if 'control' in hue:
            patch.set_edgecolor('blue')  # Blue edge for control superficial
        elif 'exp' in hue:
            patch.set_edgecolor('cyan')  # Cyan edge for exp superficial
    else:
        if 'control' in hue:
            patch.set_edgecolor('blue')  # Blue edge for control deep
        elif 'exp' in hue:
            patch.set_edgecolor('cyan')  # Cyan edge for exp deep

# # Set custom x-axis ticks
# ax3.set_xticks([0, 0.5])  # Assuming two groups, mapped to new positions
# ax3.set_xticklabels(['control', 'exp'])  # Replace with actual group names if different
# ax3.set_xlim(0,1)  # Extend x-axis to include 1, 1.7, 3
# #ax3.set_xticks([1.3, 1.7])  # Custom tick positions

ax3.set_title('Mean soma position by group and depth')
ax3.set_xlabel('Group')
ax3.set_ylabel('Mean soma position relative to the ripple peak')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.get_legend().remove()  # Remove the legend as requested
ax3.set_xlim(-1,1)
# Save the figure
# plt.savefig(fr'Q:\sachuriga\CR_CA1_paper\Results\DeepVSsuperficial/adjusted_y_hist_cdf_barplot.eps', 
#             format='eps', bbox_inches='tight')

plt.tight_layout()
plt.show()

In [ ]:
temp

In [ ]:
combined_df.to_pickle(r"Q:\sachuriga\CR_CA1_paper\tables/4depth.pkl")


In [ ]:
fig=plt.figure(figsize=(2,3))
ax3=fig.add_subplot(111)
combined_df=pd.read_pickle(r"Q:\sachuriga\CR_CA1_paper\tables/4depth.pkl")
# Create barplot with adjusted width to reduce distance
sns.barplot(data=combined_df, x='group_ani', y=y_pos, hue='group_depth', 
            palette=palette, ax=ax3, width=1)

# Customize bars for superficial layers (no face color, only edge)
for patch in ax3.patches:
    hue = ax3.get_legend().get_texts()[ax3.patches.index(patch) % len(ax3.get_legend().get_texts())].get_text()
    if 'superficial' in hue:
        patch.set_facecolor('#B755E1')  # Remove face color
        if 'control' in hue:
            patch.set_edgecolor('blue')  # Blue edge for control superficial
        elif 'exp' in hue:
            patch.set_edgecolor('red')  # Cyan edge for exp superficial
    else:
        patch.set_facecolor('#FFBB41')
        if 'control' in hue:
            patch.set_edgecolor('blue')  # Blue edge for control deep
        elif 'exp' in hue:
            patch.set_edgecolor('red')  # Cyan edge for exp deep

#ax3.set_title('Mean soma position by group and depth')
ax3.set_xlabel([]).set_visible(False)
ax3.set_ylabel('μm')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.get_legend().remove()  # Remove the legend as requested

#ax3.set_xticks = ([0.2,0.8])
ax3.set_xticklabels(["CRs +","CRs -"],rotation=-30)
ax3.set_xlim(-1,2)
# Save the figure
# plt.savefig(fr'Q:\sachuriga\CR_CA1_paper\Results\DeepVSsuperficial/adjusted_y_hist_cdf_barplot.eps', 
#             format='eps', bbox_inches='tight')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax3 = plt.subplots(1, 1, figsize=(22, 6))
# Modified Barplot (right subplot)
# Calculate the counts for deep and superficial in each group
counts = combined_df.groupby(['group_ani', 'depth']).size().unstack(fill_value=0)
counts['total'] = counts['deep'] + counts['superficial']
counts['deep_ratio'] = counts['deep'] / counts['total'] * 100
counts['superficial_ratio'] = counts['superficial'] / counts['total'] * 100

# Prepare data for the bar plot
groups = counts.index
deep_ratios = counts['deep_ratio']
superficial_ratios = counts['superficial_ratio']

# Define colors for each group
colors = {'control': 'blue', 'exp': 'cyan'}  # Colors inspired by the image
# Plot filled bars for deep
ax3.bar(x, deep_ratios, 0.8, label='pyramidal', color=[colors[g] for g in groups], alpha=0.6)

# Plot outline bars for superficial (stacked on top of deep)
ax3.bar(x, superficial_ratios, 0.8, bottom=deep_ratios, 
        edgecolor=[colors[g] for g in groups], linewidth=2, fill=False, label='interneuron')

# Add percentage labels
for i, group in enumerate(groups):
    # Deep percentage
    deep_height = deep_ratios[i]
    ax3.text(i, deep_height/2, f'{deep_height:.0f}%', ha='center', va='center', fontsize=12, fontweight='bold')
    
    # Superficial percentage
    superficial_height = superficial_ratios[i]
    total_height = deep_height + superficial_height
    ax3.text(i, deep_height + superficial_height/2, f'{superficial_height:.0f}%', 
             ha='center', va='center', fontsize=12, fontweight='bold')

# Customize the plot
ax3.set_xticks(x)
ax3.set_xticklabels(groups)
ax3.set_ylim(0, 100)  # Since we're plotting percentages
ax3.set_title('Proportion of Deep vs Superficial by Group')
ax3.set_xlabel('Group')
ax3.set_ylabel('Percentage (%)')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

folder_path = r"S:\Sachuriga\file_with_table\ripple_ch"

# Function to get pickle files
def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

# Define group prefixes
target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']

# Get the list of pickle files
pkl_files = get_pkl_files(folder_path)

# Initialize a list to store all DataFrames
all_dfs = []

# Process each pickle file
for file in pkl_files:
    df = pd.read_pickle(os.path.join(folder_path, file))
    df = df[(df['cell_type'] == "pyramidal")]
    if df.empty:
        print(f"Warning: File {file} has no pyramidal cells")
        continue
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue
    all_dfs.append(df)

# Concatenate all DataFrames into one
combined_df = pd.concat(all_dfs, ignore_index=True)

# Define the y-axis column variable
y_column = "addjust y r2"

# Calculate 5th and 95th quantiles for the 90% range
control_data = combined_df[combined_df['group_ani'] == 'control'][y_column].dropna()
exp_data = combined_df[combined_df['group_ani'] == 'exp'][y_column].dropna()

control_q5 = np.percentile(control_data, 5)
control_q95 = np.percentile(control_data, 95)
exp_q5 = np.percentile(exp_data, 5)
exp_q95 = np.percentile(exp_data, 95)

print(f"Control Quantiles: 5% = {control_q5:.2f}, 95% = {control_q95:.2f}")
print(f"Experimental Quantiles: 5% = {exp_q5:.2f}, 95% = {exp_q95:.2f}")

# Filter data to keep only values within the 90% range
control_filtered = combined_df[
    (combined_df['group_ani'] == 'control') & 
    (combined_df[y_column] >= control_q5) & 
    (combined_df[y_column] <= control_q95)
]
exp_filtered = combined_df[
    (combined_df['group_ani'] == 'exp') & 
    (combined_df[y_column] >= exp_q5) & 
    (combined_df[y_column] <= exp_q95)
]

# Combine filtered data
filtered_df = pd.concat([control_filtered, exp_filtered], ignore_index=True)

# Collect the specified columns for each group from filtered data
control_data = filtered_df[filtered_df['group_ani'] == 'control'][
    [y_column, 'matlab_test_stat_si', 'matlab_sparsity', 'matlab_maxfsize', 'bursting_index', 'mean_firing_rate']
]
exp_data = filtered_df[filtered_df['group_ani'] == 'exp'][
    [y_column, 'matlab_test_stat_si', 'matlab_sparsity', 'matlab_maxfsize', 'bursting_index', 'mean_firing_rate']
]

# Convert to lists
control_addjust_y = control_data[y_column].to_list()
control_matlab_test_stat_si = control_data['matlab_test_stat_si'].to_list()
control_matlab_sparsity = control_data['matlab_sparsity'].to_list()
control_matlab_maxfsize = control_data['matlab_maxfsize'].to_list()
control_mean_firing_rate = control_data['mean_firing_rate'].to_list()
control_bursting_index = control_data['bursting_index'].to_list()

exp_addjust_y = exp_data[y_column].to_list()
exp_matlab_test_stat_si = exp_data['matlab_test_stat_si'].to_list()
exp_matlab_sparsity = exp_data['matlab_sparsity'].to_list()
exp_matlab_maxfsize = exp_data['matlab_maxfsize'].to_list()
exp_mean_firing_rate = exp_data['mean_firing_rate'].to_list()
exp_bursting_index = exp_data['bursting_index'].to_list()

# Reconstruct DataFrames from lists for plotting
control_df = pd.DataFrame({
    y_column: control_addjust_y,
    'matlab_test_stat_si': control_matlab_test_stat_si,
    'matlab_sparsity': control_matlab_sparsity,
    'matlab_maxfsize': control_matlab_maxfsize,
    'mean_firing_rate': control_mean_firing_rate,
    'bursting_index': control_bursting_index
})
exp_df = pd.DataFrame({
    y_column: exp_addjust_y,
    'matlab_test_stat_si': exp_matlab_test_stat_si,
    'matlab_sparsity': exp_matlab_sparsity,
    'matlab_maxfsize': exp_matlab_maxfsize,
    'mean_firing_rate': exp_mean_firing_rate,
    'bursting_index': exp_bursting_index
})

# Variables to analyze
variables = ['matlab_test_stat_si', 'matlab_sparsity', 'matlab_maxfsize', 'mean_firing_rate', 'bursting_index']
titles = ['Test Stat SI', 'Sparsity', 'Max F Size', 'Mean Firing Rate', 'Bursting Index']

# Calculate correlations and store them
correlations = {}
for var, title in zip(variables, titles):
    # Control group correlation
    control_clean = control_df[[var, y_column]].dropna()
    control_corr, control_p = pearsonr(control_clean[var], control_clean[y_column])
    
    # Experimental group correlation
    exp_clean = exp_df[[var, y_column]].dropna()
    exp_corr, exp_p = pearsonr(exp_clean[var], exp_clean[y_column])
    
    correlations[var] = {
        'control_r': control_corr,
        'control_p': control_p,
        'exp_r': exp_corr,
        'exp_p': exp_p
    }
    
    print(f"\n{title} (Filtered 90% range):")
    print(f"Control Group: r = {control_corr:.3f}, p-value = {control_p:.3f}")
    print(f"Experimental Group: r = {exp_corr:.3f}, p-value = {exp_p:.3f}")

# Set up the plotting
sns.set(style="white")
fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharey=True)
axes = axes.flatten()

# Define x-axis limits for each variable
x_limits = {
    'matlab_test_stat_si': (0, 4),
    'matlab_sparsity': (0, 1.0),
    'matlab_maxfsize': (0, 400),
    'mean_firing_rate': (0, 8),
    'bursting_index': (0, 5)
}

# Plot each variable with regression lines and add statistics
for ax, var, title in zip(axes[:len(variables)], variables, titles):
    sns.regplot(x=var, y=y_column, data=control_df, ax=ax, color='blue', 
                label='Control', scatter_kws={'alpha': 0.5, "s": 5}, line_kws={'lw': 2})
    sns.regplot(x=var, y=y_column, data=exp_df, ax=ax, color='cyan', 
                label='Exp', scatter_kws={'alpha': 0.5, "s": 5}, line_kws={'lw': 2})
    ax.set_title(f'Correlation with {title} (90% Range)')
    ax.set_xlabel(title)
    ax.set_ylabel('Adjust Y Median')
    ax.legend()
    
    # Set x-limits
    ax.set_xlim(x_limits[var][0], x_limits[var][1])
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add correlation stats
    stats_text = (f"Control: r={correlations[var]['control_r']:.2f}, p={correlations[var]['control_p']:.3f}\n"
                  f"Exp: r={correlations[var]['exp_r']:.2f}, p={correlations[var]['exp_p']:.3f}")
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=8, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Remove the extra subplot
if len(variables) < len(axes):
    fig.delaxes(axes[-1])

fig.savefig(fr'Q:\sachuriga\CR_CA1_paper\Results\DeepVSsuperficial/deepVSsuperficial_90percent_range.eps', 
            format='eps', bbox_inches='tight')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
file_path = r"Q:\sachuriga\Sachuriga_Python/quattrocolo-nwb4fp/ASSY-236-F.prb"

# Read the file and parse the dictionary
local_vars = {'np': np}
with open(file_path, 'r') as file:
    exec(file.read(), local_vars)  # Execute the file content with NumPy in scope

    
channel_groups = local_vars.get('channel_groups')
if channel_groups is None:
    raise ValueError(f"'channel_groups' not found in {file_path}")

# Assuming channel_groups is loaded from Step 1
data = []
for group_id, group_data in channel_groups.items():
    channels = group_data['channels']
    geometry = group_data['geometry']
    for channel in channels:
        x, y = geometry[channel]
        data.append({
            'group_id': group_id,
            'channel_id': channel,
            'x': x,
            'y': y
        })
probe_df = pd.DataFrame(data)


for i in range(64):
    ch = i
    plt.scatter(probe_df[probe_df['channel_id']==ch]['x'], probe_df[probe_df['channel_id']==ch]['y'], marker='o',color='black',s=0.5)  # Using scatter instead of plot to show dots
    #plt.scatter(probe_df[probe_df['channel_id']==median_list[i]]['x'], probe_df[probe_df['channel_id']==median_list[i]]['y'], marker='^')
    plt.text(probe_df[probe_df['channel_id']==ch]['x'], probe_df[probe_df['channel_id']==ch]['y'], fr"ch:{i}", fontsize=8, color='black')

# df_good = df_unit_table[df_unit_table['unit_quality']=="good"]
# df_py = df_good[df_good['cell_type']=="Pyramidal cells"]

df_py = pd.read_pickle(r"S:\Sachuriga\file_with_table\ripple_ch/63383_2024-07-10_15-37-51_A_units_table_withDLC.pkl")
#df_py = df_list_actual
df_py = df_py[df_py['buzaki_py_cell_type']=="pyramidal"]
df=df_py
x=df_py['x']
y=df_py['addjust y']
hue = df_py['matlab_test_stat_si']
hot_cmap = plt.get_cmap('hot')
# Create the scatter plot
sns.scatterplot(x=x, y=y, hue=hue, palette=hot_cmap,s=100)

# Optional: Add labels and title
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Scatter Plot with Matlab Test Stat SI')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
file_path = r"Q:\sachuriga\Sachuriga_Python/quattrocolo-nwb4fp/ASSY-236-F.prb"

# Read the file and parse the dictionary
local_vars = {'np': np}
with open(file_path, 'r') as file:
    exec(file.read(), local_vars)  # Execute the file content with NumPy in scope

    
channel_groups = local_vars.get('channel_groups')
if channel_groups is None:
    raise ValueError(f"'channel_groups' not found in {file_path}")

# Assuming channel_groups is loaded from Step 1
data = []
for group_id, group_data in channel_groups.items():
    channels = group_data['channels']
    geometry = group_data['geometry']
    for channel in channels:
        x, y = geometry[channel]
        data.append({
            'group_id': group_id,
            'channel_id': channel,
            'x': x,
            'y': y
        })
probe_df = pd.DataFrame(data)


for i in range(64):
    ch = i
    plt.scatter(probe_df[probe_df['channel_id']==ch]['x'], probe_df[probe_df['channel_id']==ch]['y'], marker='o',color='black',s=0.5)  # Using scatter instead of plot to show dots
    #plt.scatter(probe_df[probe_df['channel_id']==median_list[i]]['x'], probe_df[probe_df['channel_id']==median_list[i]]['y'], marker='^')
    plt.text(probe_df[probe_df['channel_id']==ch]['x'], probe_df[probe_df['channel_id']==ch]['y'], fr"ch:{i}", fontsize=8, color='black')

# df_good = df_unit_table[df_unit_table['unit_quality']=="good"]
# df_py = df_good[df_good['cell_type']=="Pyramidal cells"]

#df_py = df_py[df_py['cell_type']=="pyramidal"]
x=df_py['x']
y=df_py['y']
hue = df_py['matlab_test_stat_si']
hot_cmap = plt.get_cmap('hot')
# Create the scatter plot
sns.scatterplot(x=x, y=y, hue=hue, palette=hot_cmap,s=100)

# Optional: Add labels and title
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Scatter Plot with Matlab Test Stat SI')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from probeinterface import Probe, ProbeGroup
from probeinterface import get_probe
import probeinterface as pi
from probeinterface.plotting import plot_probe, plot_probe_group

fig = plt.figure()
df= pd.read_pickle(r"S:\Sachuriga\file_with_table\ripple_ch/63383_2024-07-25_12-57-40_A_units_table_withDLC.pkl")
#df_py = df_list_actual
df = df[df['buzaki_py_cell_type']=="pyramidal"]
# Load probe
manufacturer = 'cambridgeneurotech'
probe_name = 'ASSY-236-F'
probe = pi.get_probe(manufacturer, probe_name)
print(probe)

# Map channels to device indices
mapping_to_device = [
    41, 39, 38, 37, 35, 34, 33, 32, 29, 30, 28, 26, 25, 24, 22, 20,
    46, 45, 44, 43, 42, 40, 36, 31, 27, 23, 21, 18, 19, 17, 16, 14,
    55, 53, 54, 52, 51, 50, 49, 48, 47, 15, 13, 12, 11, 9, 10, 8,
    63, 62, 61, 60, 59, 58, 57, 56, 7, 6, 5, 4, 3, 2, 1, 0
]
probe.set_device_channel_indices(mapping_to_device)

# Create probe dataframe
probe_df = probe.to_dataframe(complete=True)
print(probe_df.loc[:, ["contact_ids", "shank_ids", "device_channel_indices"]])

# Create probegroup and plot probe
probegroup = ProbeGroup()
probegroup.add_probe(probe)
plot_probe(probe, contacts_colors="grey")

# Assuming df is your dataframe with x, y coordinates
df_py = df[df['addjust y r2']>0]  # Replace with your actual dataframe
x = df_py['x']
y = df_py['y']
# Create scatter plot with triangle markers
# First scatter plot (blue triangles, edge only)
sns.scatterplot(x=x, y=y, s=100, alpha=1, marker='^', edgecolor='orange', facecolor='none')

# Assuming df is your dataframe with x, y coordinates
df_py = df[df['addjust y r2']<=0]  # Replace with your actual dataframe
x = df_py['x']
y = df_py['y']
# Second scatter plot (black triangles, edge only)
sns.scatterplot(x=x, y=y, s=100, alpha=1, marker='^', edgecolor='darkviolet', facecolor='none')
# Plot short horizontal lines for specified channels
chs = [20, 6, 28, 53, 44]
line_length = 100  # Adjust the length of the horizontal line as needed

for ch in chs:
    temp = probe_df[probe_df['device_channel_indices'] == ch]
    if not temp.empty:
        x, y = temp['x'].iloc[0], temp['y'].iloc[0]
        # Draw a short horizontal line centered at (x, y)
        plt.plot([x - line_length/2, x + line_length/2], [y, y], 'k--', linewidth=1)

plt.axis('off')  # Hides the entire axis (including ticks, labels, and spines)
plt.title('')    # Ensures no title is displayed

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np

df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')

df_loaded['buzaki_cell_type']=None

for i in range(len(df_loaded)):
    if df_loaded['peak_to_valley'].iloc[i] <= 0.000425:
        df_loaded['buzaki_cell_type'].iloc[i] = "narrow_spike_interneurons"
    elif  (df_loaded['peak_to_valley'].iloc[i] > 0.000425) & (df_loaded['matlab_acg_tau_rise_1'].iloc[i]> 6):
        df_loaded['buzaki_cell_type'].iloc[i] = "wide_spike_interneurons"
    else:
        df_loaded['buzaki_cell_type'].iloc[i] = "pyramidal"

df_loaded.to_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/good_units_with_tsnLabels.pkl')

In [ ]:
pip install probeinterface

In [ ]:
import pandas as pd
df = pd.read_pickle(r"S:\Sachuriga\file_with_table\ripple_ch/63383_2024-07-25_12-57-40_units_table_withDLC.pkl")
df